In [1]:
"""
Demo: Two-Spool Turbofan (Fuel Flow Mode)

This script runs a Huracan two-spool turbofan controlled by fuel flow.
- Engine defined by make_engine_two_spool (must be available in your environment).
- Inputs: altitude, Mach, ISA offset, fuel flow.
- Outputs: thrust, SFC, TSFC, and station data.
- Includes simple plots.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from huracan.gas import gas
from huracan.engine import system, isa_tp


# --- Helpers ---
def isa_tp_offset(h_m, dT_C=0.0):
    T, p = isa_tp(h_m)
    return T + dT_C, p

def stream_totals(s):
    return float(s.gas.t_0), float(s.gas.p0)

def safe_v_exit_from_temperatures(T0_exit, T_amb, cp):
    dT = max(1e-6, T0_exit - T_amb)
    return np.sqrt(2 * cp * dT)

def tsfc_lbm_hr_lbf(thrust_N, fuel_kgps):
    if thrust_N <= 0:
        return np.nan
    return (fuel_kgps * 2.20462 * 3600) / (thrust_N * 0.224809)

# --- Single condition runner (fuel-flow only) ---
def run_one_condition(
    h_m, M, dT_isa_C,
    mf_fuel,
    mdot_air=520.0,
    bpr=5.2, pi_fan=1.52, pi_lpc=2.0, pi_hpc=12.0,
    eta_turb=0.96
):
    # Engine: must already include fuel hookup
    engbits = make_engine_two_spool(
        bpr=bpr, pi_fan=pi_fan, pi_lpc=pi_lpc, pi_hpc=pi_hpc,
        eta_turb=eta_turb, mf_fuel=mf_fuel
    )
    s_fan, s_core, s_byp = engbits['s_fan'], engbits['s_core'], engbits['s_byp']
    Comb = engbits['Comb']

    # Ambient
    T_amb, p_amb = isa_tp_offset(h_m, dT_isa_C)
    g = gas(mf=mdot_air, m=M, t_0=T_amb, p_0=p_amb)
    g - s_fan

    # Run
    eng = system(s_core, s_byp)
    eng.run(log=False)

    # Manual thrust (core + bypass)
    cp_core = s_core.gas.cp(s_core.gas.t_0)
    cp_byp  = s_byp.gas.cp(s_byp.gas.t_0)
    V_exit_core = safe_v_exit_from_temperatures(s_core.gas.t_0, T_amb, cp_core)
    V_exit_byp  = safe_v_exit_from_temperatures(s_byp.gas.t_0,  T_amb, cp_byp)
    thrust_core = s_core.gas.mf * (V_exit_core - s_core.gas.v_0)
    thrust_byp  = s_byp.gas.mf  * (V_exit_byp  - s_byp.gas.v_0)
    thrust = thrust_core + thrust_byp

    fuel = float(Comb.fuel.mf) if getattr(Comb.fuel, 'mf', None) else 0.0
    sfc  = fuel / max(1e-9, thrust)
    tsfc = tsfc_lbm_hr_lbf(thrust, fuel)

    perf = {
        'h_m': h_m, 'M': M, 'dT_isa_C': dT_isa_C,
        'thrust_N': thrust, 'fuel_kgps': fuel,
        'sfc_kg_per_Ns': sfc, 'tsfc_lbm_hr_lbf': tsfc
    }
    stn = {
        'h_m': h_m, 'M': M, 'dT_isa_C': dT_isa_C,
        'T0_fan': float(s_fan.gas.t_0), 'P0_fan': float(s_fan.gas.p0),
        'T0_core': float(s_core.gas.t_0), 'P0_core': float(s_core.gas.p0),
        'T_ambient_K': T_amb
    }
    return perf, stn

# --- Sweep grid ---
def sweep_grid(alts_m, machs, dT_list_C, fuel_list, **engine_kwargs):
    perf_rows, stn_rows = [], []
    for h in alts_m:
        for M in machs:
            for dT in dT_list_C:
                for mf in fuel_list:
                    perf, stn = run_one_condition(h, M, dT, mf_fuel=mf, **engine_kwargs)
                    perf_rows.append(perf)
                    stn_rows.append(stn)
    return pd.DataFrame(perf_rows), pd.DataFrame(stn_rows)

# === Example run ===
if __name__ == "__main__":
    profile_alts  = [0, 3000, 9000, 11000]
    profile_machs = [0.25, 0.45, 0.78, 0.80]
    dTs   = [0]
    fuels = [0.8, 1.0, 1.2]  # kg/s fuel flow

    perf_df, stn_df = sweep_grid(
        profile_alts, profile_machs, dTs, fuels,
        mdot_air=520.0, bpr=5.2, pi_fan=1.52, pi_lpc=2.0, pi_hpc=12.0, eta_turb=0.96
    )

    print(perf_df.head())
    print(stn_df.head())

    # --- Plotting ---
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # Thrust vs altitude for each fuel flow (at M=0.8)
    for mf in fuels:
        df = perf_df[(perf_df['M']==0.8) & (perf_df['fuel_kgps'].round(2)==mf)]
        ax[0].plot(df['h_m'], df['thrust_N'], marker='o', label=f"mf={mf} kg/s")
    ax[0].set_title("Thrust vs Altitude (M=0.8)")
    ax[0].set_xlabel("Altitude [m]")
    ax[0].set_ylabel("Thrust [N]")
    ax[0].legend()

    # TSFC vs altitude for each fuel flow (at M=0.8)
    for mf in fuels:
        df = perf_df[(perf_df['M']==0.8) & (perf_df['fuel_kgps'].round(2)==mf)]
        ax[1].plot(df['h_m'], df['tsfc_lbm_hr_lbf'], marker='s', label=f"mf={mf} kg/s")
    ax[1].set_title("TSFC vs Altitude (M=0.8)")
    ax[1].set_xlabel("Altitude [m]")
    ax[1].set_ylabel("TSFC [lbm/hr/lbf]")
    ax[1].legend()

    plt.tight_layout()
    plt.show()


ModuleNotFoundError: No module named 'huracan.gas'

In [ ]:
%run Demo_TwoSpool_FuelFlow.py